# Paper tables

Reproduces every table in the paper from saved model outputs, without retraining:
- **Classical models** (logistic regression, random forest, XGBoost) are refit here from the cached similarity features (deterministic, about a minute; the first run builds the feature cache, a few minutes).
- **Neural models** are read from `predictions/<run>/probs_{val,test}_seed_{s}.npy`, written by each family's training notebook. A prediction is probability ≥ 0.5.

Needs the pair files (`pipeline/`, or `load_pairs` pointing at them). Each section header names the paper table it produces.

| Section | Paper table |
|---|---|
| 1 | Table 1 (pair types, test counts) |
| 2 | Tables 3, 4 (classical, sentinel vs zero-fill), A1, A2 (random-forest importance) |
| 3 | Tables 5, 6, 7 (siamese BiLSTM) |
| 4 | Tables 8, 9, 10, 11 (ESIM) |
| 5 | Tables 12, 13, 14 (DistilBERT, cross-family AUC) |
| 6 | Discussion: AUC of matches vs case 1 within the missing-number group, every model |

## 0. Setup

In [1]:
import os
import sys
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from helpers.pairs import load_pairs, add_number_street
from helpers.paths import FEATURES, PREDICTIONS, probs_path
from helpers.features import get_processed_dfs
from classical.models import run_against_models

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

DATASET = "overture_case2s"
THRESHOLD = 0.5
SEEDS = [0, 1, 2]

pairs = {split: load_pairs(DATASET, split) for split in ["train", "val", "test"]}
df = {split: add_number_street(d) for split, (d, _) in pairs.items()}
y = {split: lab.values for split, (_, lab) in pairs.items()}

In [2]:
# Pair-type masks (one per row of the paper's per-type tables) and the house-number groups
SLICES = {   # name -> (neg_type selector, "found" for matches / "rejected" for non-matches)
    "Identical found":     lambda d, l: (l == 1) & (d["address_a"] == d["address_b"]).values,
    "Non-identical found": lambda d, l: (l == 1) & (d["address_a"] != d["address_b"]).values,
    "Case 1 rejected":     lambda d, l: (d["neg_type"] == "case1").values,
    "Case 2s rejected":    lambda d, l: (d["neg_type"] == "case2s").values,
    "Case 3 rejected":     lambda d, l: (d["neg_type"] == "case3").values,
}
masks = {split: {name: f(df[split], y[split]) for name, f in SLICES.items()} for split in df}
one_missing = {split: (d["number_a"].notna() != d["number_b"].notna()).values for split, d in df.items()}


def pct_correct(preds, split, mask):
    return 100 * (preds[mask] == y[split][mask]).mean()


def slice_row(probs, split):
    """% correct per pair type + AUC non-identical vs case 2s, for one run's probabilities."""
    preds = (probs >= THRESHOLD).astype(int)
    row = {name: pct_correct(preds, split, m) for name, m in masks[split].items()}
    row["AUC non-identical vs case 2s"] = auc_vs(probs, masks[split]["Non-identical found"], masks[split]["Case 2s rejected"])
    return row


def auc_vs(probs, pos_mask, neg_mask):
    """AUC: chance that a random pair of pos_mask scores above a random pair of neg_mask."""
    return roc_auc_score(np.r_[np.ones(pos_mask.sum()), np.zeros(neg_mask.sum())], np.r_[probs[pos_mask], probs[neg_mask]])


def load_probs(run, split, seed):
    return np.load(probs_path(run, split, seed)).ravel()


def fmt_range(values, digits=1):
    """Min–max over seeds, as in the paper ('a–b', or one number when they agree)."""
    lo, hi = f"{min(values):.{digits}f}", f"{max(values):.{digits}f}"
    return lo if lo == hi else f"{lo}–{hi}"


def range_table(rows_per_model, digits=None):
    """{model: [row per seed]} -> one formatted row per model (min–max over seeds per column)."""
    out = {}
    for model, rows in rows_per_model.items():
        frame = pd.DataFrame(rows)
        out[model] = {col: fmt_range(frame[col], 3 if col.startswith("AUC") else 1) for col in frame.columns}
    return pd.DataFrame(out).T

## 1. Table 1: pair types, test counts

In [3]:
test = df["test"]
types = pd.Series({
    "Positive, identical": masks["test"]["Identical found"].sum(),
    "Positive, non-identical": masks["test"]["Non-identical found"].sum(),
    "Case 1": (test["neg_type"] == "case1").sum(),
    "Case 2": (test["neg_type"] == "case2").sum(),
    "Case 2s": (test["neg_type"] == "case2s").sum(),
    "Case 3": (test["neg_type"] == "case3").sum(),
}, name="test n")
print("case 1 pairs with a number on one side only:", int((one_missing["test"] & (test["neg_type"] == "case1").values).sum()))
types.map("{:,}".format).to_frame()

case 1 pairs with a number on one side only: 344


,test n
"Positive, identical","3,543"
"Positive, non-identical",843
Case 1,"14,448"
Case 2,15
Case 2s,"1,053"
Case 3,"23,942"


## 2. Classical models
Features: `helpers/features.py` (23 features; `missing_number_as_zero=False` = **sentinel**, `True` = **zero-fill**). Models: `classical/models.py`, library defaults, fit on the case 2s train split.

In [4]:
os.makedirs(FEATURES, exist_ok=True)
X, X_zero = {}, {}
X["train"], X["val"], X["test"] = get_processed_dfs([[df[s], f"{FEATURES}/{s}.parquet"] for s in ["train", "val", "test"]])
X_zero["train"], X_zero["val"], X_zero["test"] = get_processed_dfs(
    [[df[s], f"{FEATURES}/{s}_missing_as_zero.parquet"] for s in ["train", "val", "test"]], missing_number_as_zero=True)

ret, fitted = run_against_models(X["train"], y["train"], X["val"], X["test"], return_models=True)
ret_zero, fitted_zero = run_against_models(X_zero["train"], y["train"], X_zero["val"], X_zero["test"], return_models=True)
SHORT = {"LogisticRegression": "LogReg", "RandomForestClassifier": "RF", "XGBClassifier": "XGB"}

### Table 3: classical models per pair type, sentinel vs zero-fill (test, % correct)

In [5]:
table3 = {}
for model, short in SHORT.items():
    for setting, r in [("sentinel", ret), ("zero-fill", ret_zero)]:
        preds = r[model]["test"][0]
        table3[f"{short} {setting}"] = {name: pct_correct(preds, "test", m) for name, m in masks["test"].items()}
table3 = pd.DataFrame(table3)
table3.insert(0, "n", [int(m.sum()) for m in masks["test"].values()])
table3.round(1)

,n,LogReg sentinel,LogReg zero-fill,RF sentinel,RF zero-fill,XGB sentinel,XGB zero-fill
Identical found,3543,99.9,99.9,100.0,100.0,100.0,100.0
Non-identical found,843,50.5,44.8,70.1,67.7,69.9,67.7
Case 1 rejected,14448,99.1,100.0,99.6,99.8,99.6,99.9
Case 2s rejected,1053,74.4,72.7,88.7,88.8,88.5,87.4
Case 3 rejected,23942,100.0,100.0,99.9,100.0,100.0,100.0


### Table 4: pairs where exactly one side has a house number, by type (test, % correct)

In [6]:
groups4 = {"Matches (found)": "case0", "Case 1 non-matches (rejected)": "case1", "Case 3 non-matches (rejected)": "case3"}
table4 = {}
for model, short in SHORT.items():
    for setting, r in [("sentinel", ret), ("zero-fill", ret_zero)]:
        preds = r[model]["test"][0]
        table4[f"{short} {setting}"] = {g: pct_correct(preds, "test", one_missing["test"] & (test["neg_type"] == t).values)
                                        for g, t in groups4.items()}
table4 = pd.DataFrame(table4)
table4.insert(0, "n", [int((one_missing["test"] & (test["neg_type"] == t).values).sum()) for t in groups4.values()])
table4.round(1)

,n,LogReg sentinel,LogReg zero-fill,RF sentinel,RF zero-fill,XGB sentinel,XGB zero-fill
Matches (found),164,33.5,0.0,27.4,11.6,22.6,11.6
Case 1 non-matches (rejected),344,66.0,100.0,82.0,95.9,84.9,96.5
Case 3 non-matches (rejected),990,99.7,100.0,98.9,100.0,99.3,100.0


### Tables A1, A2: random-forest feature importance (mean impurity decrease, sums to 1), sentinel vs zero-fill
Feature names: `*_address` / `*_address_sim` = house number, `*_street` = street, `*_zip` / `*_zipcode_sim` = ZIP. A1 sums the unrounded per-feature importances of A2.

In [7]:
COMPONENT = lambda f: ("Flag" if f == "number_missing" else "House number" if "address" in f
                       else "Street" if "street" in f else "ZIP")
METRIC = {"lv": "Levenshtein", "jw": "Jaro-Winkler", "hm": "Hamming", "jaccard": "Jaccard", "cosine": "Cosine",
          "tversky": "Tversky", "sorenson": "Sørensen–Dice", "overlap": "Overlap", "number": "number_missing"}
importance = pd.DataFrame({
    setting: pd.Series(f["RandomForestClassifier"].feature_importances_, index=f["RandomForestClassifier"].feature_names_in_)
    for setting, f in [("RF sentinel", fitted), ("RF zero-fill", fitted_zero)]
})
importance.index.name = "feature"
tableA2 = importance.assign(Component=importance.index.map(COMPONENT),
                            Metric=importance.index.map(lambda f: METRIC[f.split("_")[0]]))
order = {"House number": 0, "Street": 1, "ZIP": 2, "Flag": 3}
tableA2 = tableA2.sort_values(["Component", "RF sentinel"], key=lambda c: c.map(order) if c.name == "Component" else -c)
tableA1 = tableA2.groupby("Component", sort=False)[["RF sentinel", "RF zero-fill"]].sum(min_count=1)
tableA1.insert(0, "features", tableA2.groupby("Component", sort=False).size())
display(tableA1.round(3))
tableA2.set_index(["Component", "Metric"])[["RF sentinel", "RF zero-fill"]].round(3)

,features,RF sentinel,RF zero-fill
Component,,,
House number,7,0.721,0.727
Street,7,0.253,0.256
ZIP,8,0.025,0.017
Flag,1,0.000,NaN


RF sentinel  RF zero-fill
Component    Metric                                   
House number Levenshtein           0.196         0.207
             Jaro-Winkler          0.127         0.121
             Jaccard               0.109         0.093
             Cosine                0.089         0.122
             Tversky               0.081         0.071
             Sørensen–Dice         0.063         0.094
             Overlap               0.057         0.017
Street       Overlap               0.065         0.061
             Jaro-Winkler          0.051         0.062
             Levenshtein           0.035         0.043
             Cosine                0.030         0.026
             Sørensen–Dice         0.027         0.023
             Tversky               0.023         0.020
             Jaccard               0.022         0.021
ZIP          Cosine                0.005         0.001
             Sørensen–Dice         0.004         0.002
             Hamming               0.004         0.001
             Levenshtein           0.003         0.002
             Jaccard               0.003         0.002
             Tversky               0.003         0.006
             Overlap               0.003         0.002
             Jaro-Winkler          0.001         0.002
Flag         number_missing        0.000           NaN

In [8]:
# XGBoost's gain importance, quoted in the caption of Table A1 (concentrates on one feature per correlated group)
xgb_importance = pd.Series(fitted["XGBClassifier"].feature_importances_, index=fitted["XGBClassifier"].feature_names_in_)
xgb_importance.sort_values(ascending=False).head(3).round(2)

jw_address_sim    0.81
overlap_street    0.06
jw_zipcode_sim    0.04
dtype: float32

## 3. Siamese BiLSTM
Runs `siamese_mean`, `siamese_max` (seed 0), from `siamese/train.ipynb`.

### Table 5: by pooling (test, %, AUC non-identical vs case 2s)

In [9]:
siamese = {pooling: load_probs(f"siamese_{pooling}", "test", 0) for pooling in ["mean", "max"]}
table5 = pd.DataFrame({f"{p.capitalize()}": slice_row(probs, "test") for p, probs in siamese.items()}).T
table5.round(3)

,Identical found,Non-identical found,Case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
Mean,100.0,80.071,99.28,75.689,99.971,0.854
Max,100.0,78.292,99.01,76.543,99.958,0.835


### Table 6: pairs where exactly one side has a house number (test)

In [10]:
m_match = one_missing["test"] & (test["neg_type"] == "case0").values
m_case1 = one_missing["test"] & (test["neg_type"] == "case1").values
table6 = {}
for p, probs in siamese.items():
    preds = (probs >= THRESHOLD).astype(int)
    table6[p.capitalize()] = {"Matches found (%)": pct_correct(preds, "test", m_match),
                              "Case 1 rejected (%)": pct_correct(preds, "test", m_case1),
                              "AUC, matches vs case 1": auc_vs(probs, m_match, m_case1)}
print(f"n: {int(m_match.sum())} matches / {int(m_case1.sum())} case 1")
pd.DataFrame(table6).T.round(3)

n: 164 matches / 344 case 1


,Matches found (%),Case 1 rejected (%),"AUC, matches vs case 1"
Mean,32.927,71.221,0.526
Max,45.732,59.012,0.545


### Table 7: siamese BiLSTM against the tree models (test)

In [11]:
classical_probs = {f"{SHORT[m]} sentinel": ret[m]["test"][1] for m in SHORT}
table7 = pd.DataFrame({
    "Random forest (sentinel)": slice_row(classical_probs["RF sentinel"], "test"),
    "XGBoost (sentinel)": slice_row(classical_probs["XGB sentinel"], "test"),
    "Siamese BiLSTM, mean pooling (1 seed)": slice_row(siamese["mean"], "test"),
}).T
table7.round(3)

,Identical found,Non-identical found,Case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
Random forest (sentinel),99.972,70.107,99.564,88.699,99.950,0.902
XGBoost (sentinel),99.972,69.870,99.619,88.509,99.967,0.902
"Siamese BiLSTM, mean pooling (1 seed)",100.000,80.071,99.280,75.689,99.971,0.854


## 4. ESIM
Runs `esim_{mc2,mc1,ft_mc1,rand_mc2,stack_mc1}`, 3 seeds each, from `esim/02_train.ipynb`. The rare-number group (same street, both house numbers outside the word2vec `min_count` 2 vocabulary) is the mask saved by that notebook's §4.

In [12]:
ESIM = {   # paper name -> run
    "word2vec, min_count 2": "esim_mc2",
    "word2vec, min_count 1": "esim_mc1",
    "fastText, min_count 1": "esim_ft_mc1",
    "Random vector per rare word": "esim_rand_mc2",
    "word2vec + fastText stacked": "esim_stack_mc1",
}
rare = {split: np.load(os.path.join(PREDICTIONS, f"rare_number_group_{split}.npy")) for split in ["val", "test"]}
rare_match = {s: rare[s] & masks[s]["Identical found"] for s in rare}
rare_case1 = {s: rare[s] & masks[s]["Case 1 rejected"] for s in rare}
for s in rare:
    print(s, f"rare-number group: {int(rare_match[s].sum())} identical matches / {int(rare_case1[s].sum())} case 1")


def rare_counts(run, split, seed):
    p = load_probs(run, split, seed)
    preds = (p >= THRESHOLD).astype(int)
    return {"matches found": int((preds[rare_match[split]] == 1).sum()),
            "non-matches rejected": int((preds[rare_case1[split]] == 0).sum()),
            "AUC": auc_vs(p, rare_match[split], rare_case1[split])}

val rare-number group: 78 identical matches / 161 case 1
test rare-number group: 178 identical matches / 404 case 1


### Table 8: word2vec `min_count` 1, seed by seed, on the rare-number group (test)

In [13]:
rows = {f"Seed {seed}": rare_counts("esim_mc1", "test", seed) for seed in SEEDS}
n_m, n_c = int(rare_match["test"].sum()), int(rare_case1["test"].sum())
pd.DataFrame({s: {"Matches found": f"{r['matches found']} ({100 * r['matches found'] / n_m:.0f}%)",
                  "Non-matches rejected": f"{r['non-matches rejected']} ({100 * r['non-matches rejected'] / n_c:.0f}%)",
                  "AUC": f"{r['AUC']:.3f}"} for s, r in rows.items()}).T

,Matches found,Non-matches rejected,AUC
Seed 0,162 (91%),149 (37%),0.761
Seed 1,72 (40%),316 (78%),0.765
Seed 2,178 (100%),64 (16%),0.706


### Table 9: embedding settings on the rare-number group (validation, % over three seeds)
The per-seed counts below the table are the ones quoted in the text (e.g. the reproduction's seeds 0 and 1 reject 141 and 158 of 161).

In [14]:
per_seed9 = {name: [rare_counts(run, "val", seed) for seed in SEEDS] for name, run in ESIM.items()}
n_m, n_c = int(rare_match["val"].sum()), int(rare_case1["val"].sum())
table9 = pd.DataFrame({name: {
    "Same address: matches found (%)": fmt_range([100 * r["matches found"] / n_m for r in rows], 0),
    "Different numbers: non-matches rejected (%)": fmt_range([100 * r["non-matches rejected"] / n_c for r in rows], 0),
} for name, rows in per_seed9.items()}).T
display(table9)
pd.DataFrame({(name, f"seed {s}"): r for name, rows in per_seed9.items() for s, r in zip(SEEDS, rows)}).T

,Same address: matches found (%),Different numbers: non-matches rejected (%)
"word2vec, min_count 2",12–100,0–98
"word2vec, min_count 1",49–100,7–69
"fastText, min_count 1",100,89–94
Random vector per rare word,100,96–98
word2vec + fastText stacked,100,78–92


matches found  non-matches rejected       AUC
word2vec, min_count 2       seed 0           29.0                 141.0  0.678850
                            seed 1            9.0                 158.0  0.562709
                            seed 2           78.0                   0.0  0.711897
word2vec, min_count 1       seed 0           72.0                  21.0  0.690715
                            seed 1           38.0                 111.0  0.775999
                            seed 2           78.0                  12.0  0.745182
fastText, min_count 1       seed 0           78.0                 151.0  1.000000
                            seed 1           78.0                 151.0  1.000000
                            seed 2           78.0                 144.0  1.000000
Random vector per rare word seed 0           78.0                 158.0  1.000000
                            seed 1           78.0                 155.0  1.000000
                            seed 2           78.0                 154.0  1.000000
word2vec + fastText stacked seed 0           78.0                 126.0  1.000000
                            seed 1           78.0                 148.0  1.000000
                            seed 2           78.0                 128.0  0.997930

### Table 10: ESIM settings by pair type (validation, min–max over three seeds)

In [15]:
rows10 = {}
for name, run in ESIM.items():
    rows10[name] = []
    for seed in SEEDS:
        row = slice_row(load_probs(run, "val", seed), "val")
        row["Rare-number case 1 rejected"] = 100 * rare_counts(run, "val", seed)["non-matches rejected"] / int(rare_case1["val"].sum())
        rows10[name].append(row)
table10 = range_table(rows10)
table10["Rare-number case 1 rejected"] = [fmt_range([r["Rare-number case 1 rejected"] for r in rows], 0) for rows in rows10.values()]
table10[["Identical found", "Non-identical found", "Case 1 rejected", "Rare-number case 1 rejected",
         "Case 2s rejected", "Case 3 rejected", "AUC non-identical vs case 2s"]]

,Identical found,Non-identical found,Case 1 rejected,Rare-number case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
"word2vec, min_count 2",97.9–100.0,80.9–82.7,98.2–99.1,0–98,76.1–80.0,99.9–100.0,0.867–0.883
"word2vec, min_count 1",98.8–100.0,78.8–81.1,98.1–98.7,7–69,72.7–83.6,100.0,0.840–0.881
"fastText, min_count 1",100.0,79.9–81.6,98.2–98.9,89–94,64.3–71.0,100.0,0.825–0.855
Random vector per rare word,100.0,77.8–81.8,99.0–99.2,96–98,74.6–82.2,100.0,0.856–0.878
word2vec + fastText stacked,100.0,78.9–80.8,98.6–99.4,78–92,78.3–84.5,99.9–100.0,0.875–0.896


### Table 11: ESIM against the siamese BiLSTM (test)
The ESIM test AUCs of the fastText and stacked settings, quoted in the text next to Table 10, are in the second table.

In [16]:
table11 = range_table({
    "Siamese BiLSTM, mean pooling (1 seed)": [slice_row(siamese["mean"], "test")],
    "ESIM, random vector per rare word (3 seeds)": [slice_row(load_probs("esim_rand_mc2", "test", s), "test") for s in SEEDS],
})
display(table11)
range_table({name: [slice_row(load_probs(run, "test", s), "test") for s in SEEDS] for name, run in ESIM.items()})

,Identical found,Non-identical found,Case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
"Siamese BiLSTM, mean pooling (1 seed)",100.0,80.1,99.3,75.7,100.0,0.854
"ESIM, random vector per rare word (3 seeds)",99.9–100.0,74.9–78.9,98.6–98.9,72.9–75.9,99.9–100.0,0.815–0.842


,Identical found,Non-identical found,Case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
"word2vec, min_count 2",95.5–100.0,77.9–80.7,96.1–98.4,73.9–75.5,99.9,0.845–0.850
"word2vec, min_count 1",96.8–100.0,76.6–78.4,95.7–98.3,72.1–79.6,99.9,0.814–0.855
"fastText, min_count 1",100.0,76.9–78.4,97.7–99.0,68.9–72.1,99.9–100.0,0.792–0.834
Random vector per rare word,99.9–100.0,74.9–78.9,98.6–98.9,72.9–75.9,99.9–100.0,0.815–0.842
word2vec + fastText stacked,100.0,75.3–78.4,97.6–99.6,77.8–80.7,99.9–100.0,0.851–0.869


## 5. DistilBERT bi- and cross-encoder, and the cross-family comparison
Runs `distilbert_bi`, `distilbert_cross`, 3 seeds each, from `distilbert/train_{bi,cross}.ipynb`.

### Table 12: bi- and cross-encoder (test, min–max over three seeds)

In [17]:
BERT = {"Bi-encoder": "distilbert_bi", "Cross-encoder": "distilbert_cross"}
range_table({name: [slice_row(load_probs(run, "test", s), "test") for s in SEEDS] for name, run in BERT.items()})

,Identical found,Non-identical found,Case 1 rejected,Case 2s rejected,Case 3 rejected,AUC non-identical vs case 2s
Bi-encoder,100.0,82.0–83.7,99.0–99.4,58.2–62.9,99.9–100.0,0.836–0.845
Cross-encoder,100.0,70.2–73.0,99.7–99.9,82.2–92.9,100.0,0.877–0.888


### Table 13: AUC of each group of non-identical matches against all case 2s pairs (test)
Groups by the house numbers of the match (`add_number_street`): **same number** = both numbers equal, or both addresses without one; **missing on one side** = exactly one address has a number; **different, both present** = two different numbers.

In [18]:
nonid, c2s = masks["test"]["Non-identical found"], masks["test"]["Case 2s rejected"]
has_a, has_b = test["number_a"].notna().values, test["number_b"].notna().values
same_number = (test["number_a"] == test["number_b"]).values | (~has_a & ~has_b)
GROUPS13 = {
    "All non-identical": nonid,
    "Same number": nonid & same_number,
    "Missing on one side": nonid & one_missing["test"],
    "Different numbers, both present": nonid & has_a & has_b & ~same_number,
    "All but missing": nonid & ~one_missing["test"],
}
models13 = {
    "Bi-encoder": [load_probs("distilbert_bi", "test", s) for s in SEEDS],
    "Cross-encoder": [load_probs("distilbert_cross", "test", s) for s in SEEDS],
    "Random forest": [classical_probs["RF sentinel"]],
    "ESIM (random vector)": [load_probs("esim_rand_mc2", "test", s) for s in SEEDS],
}
table13 = pd.DataFrame({model: {g: fmt_range([auc_vs(p, m, c2s) for p in probs_list], 3) for g, m in GROUPS13.items()}
                        for model, probs_list in models13.items()})
table13.insert(0, "n", [int(m.sum()) for m in GROUPS13.values()])
table13

,n,Bi-encoder,Cross-encoder,Random forest,ESIM (random vector)
All non-identical,843,0.836–0.845,0.877–0.888,0.902,0.815–0.842
Same number,638,0.925–0.934,0.922–0.927,0.923,0.880–0.906
Missing on one side,164,0.448–0.497,0.689–0.729,0.845,0.547–0.667
"Different numbers, both present",41,0.949–0.963,0.928–0.940,0.798,0.808–0.853
All but missing,679,0.927–0.935,0.923–0.928,0.916,0.878–0.903


### Table 14: every model on the same AUC, non-identical matches vs case 2s (test)

In [19]:
models14 = {
    "XGBoost / random forest": [classical_probs["XGB sentinel"], classical_probs["RF sentinel"]],
    "DistilBERT cross-encoder": models13["Cross-encoder"],
    "Siamese BiLSTM, mean pooling": [siamese["mean"]],
    "DistilBERT bi-encoder": models13["Bi-encoder"],
    "ESIM, random vector per rare word": models13["ESIM (random vector)"],
}
pd.DataFrame({"AUC, non-identical vs case 2s": {model: fmt_range([auc_vs(p, nonid, c2s) for p in probs_list], 3)
                                                 for model, probs_list in models14.items()}})

,"AUC, non-identical vs case 2s"
XGBoost / random forest,0.902
DistilBERT cross-encoder,0.877–0.888
"Siamese BiLSTM, mean pooling",0.854
DistilBERT bi-encoder,0.836–0.845
"ESIM, random vector per rare word",0.815–0.842


## 6. Discussion: matches vs case 1 within the missing-number group
Pairs where exactly one side has a house number (`74 BEACH RD` / `BEACH RD`): AUC of the matches against the same-street case 1 non-matches, every model scored (test).

In [20]:
models_disc = {
    **{f"{SHORT[m]} sentinel": [ret[m]["test"][1]] for m in SHORT},
    "Siamese mean": [siamese["mean"]], "Siamese max": [siamese["max"]],
    **{f"ESIM {name}": [load_probs(run, "test", s) for s in SEEDS] for name, run in ESIM.items()},
    "Bi-encoder": models13["Bi-encoder"], "Cross-encoder": models13["Cross-encoder"],
}
pd.DataFrame({"AUC, matches vs case 1 (one side missing)": {
    model: fmt_range([auc_vs(p, m_match, m_case1) for p in probs_list], 3) for model, probs_list in models_disc.items()}})

,"AUC, matches vs case 1 (one side missing)"
LogReg sentinel,0.500
RF sentinel,0.556
XGB sentinel,0.537
Siamese mean,0.526
Siamese max,0.545
"ESIM word2vec, min_count 2",0.620–0.624
"ESIM word2vec, min_count 1",0.594–0.617
"ESIM fastText, min_count 1",0.603–0.619
ESIM Random vector per rare word,0.586–0.631
ESIM word2vec + fastText stacked,0.589–0.638
